In [0]:
# COMMAND ----------
# DBTITLE 1, Imports and config

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

PREDICTIONS_DIR = "/dbfs/eval_predictions"
FIGURES_DIR     = "/dbfs/thesis_figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

HORIZONS = ["4h", "24h"]
MODELS   = [("LogisticRegression", "LogisticRegression"), ("XGBoost", "XGBoost"), ("LSTM", "LSTM")]
# (file_prefix, display_name)

# COMMAND ----------
# DBTITLE 1, Loading and confusion matrix helpers

def load_predictions(file_prefix, horizon):
    import pandas as pd, json
    label_col = f"label_{horizon}"
    df = pd.read_parquet(f"{PREDICTIONS_DIR}/{file_prefix}_{label_col}.parquet")
    with open(f"{PREDICTIONS_DIR}/{file_prefix}_{label_col}_threshold.json") as f:
        thr = json.load(f)["optimal_threshold"]
    return df["y_true"].values, df["y_prob"].values, thr

def build_cm(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.where(row_sums > 0, cm / row_sums, 0.0)
    return cm, cm_norm

def fmt_count(n):
    return f"{int(n):,}"

# COMMAND ----------
# DBTITLE 1, Single-panel renderer

def plot_cm_panel(ax, cm_counts, cm_norm, title, cmap, vmax=1.0):
    im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=vmax, aspect="equal")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("Actual",    fontsize=10)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["No overload", "Overload"], fontsize=9)
    ax.set_yticklabels(["No overload", "Overload"], fontsize=9, rotation=90, va="center")
    
    # Cell text: count on top, proportion below
    for i in range(2):
        for j in range(2):
            count = cm_counts[i, j]
            prop  = cm_norm[i, j]
            # Choose text colour based on cell darkness
            text_color = "white" if prop > 0.5 else "black"
            ax.text(j, i, fmt_count(count),
                    ha="center", va="center",
                    fontsize=10, color=text_color, fontweight="bold")
            ax.text(j, i + 0.22, f"({prop:.3f})",
                    ha="center", va="center",
                    fontsize=9, color=text_color)
    return im

# COMMAND ----------
# DBTITLE 1, Three-panel figure builder

def plot_three_panel_cm(horizon, threshold_mode):
    """
    threshold_mode: "default" (0.5) or "optimal" (per-model F1-optimal).
    """
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), constrained_layout=True)
    cmap = plt.cm.Blues
    
    im_last = None
    for ax, (file_prefix, display) in zip(axes, MODELS):
        y_true, y_prob, optimal = load_predictions(file_prefix, horizon)
        thr = 0.5 if threshold_mode == "default" else optimal
        cm_counts, cm_norm = build_cm(y_true, y_prob, thr)
        title = f"{display} {horizon} — threshold = {thr:.2f}"
        im_last = plot_cm_panel(ax, cm_counts, cm_norm, title, cmap)
    
    # Shared colorbar on the right
    cbar = fig.colorbar(im_last, ax=axes, fraction=0.035, pad=0.02)
    cbar.set_label("Row-normalised proportion", fontsize=10)
    
    fname = f"thesis_cm_{horizon}_{threshold_mode}.png"
    out_path = f"{FIGURES_DIR}/{fname}"
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return fig

# COMMAND ----------
# DBTITLE 1, Generate all 4 figures

for horizon in HORIZONS:
    for mode in ["default", "optimal"]:
        fig = plot_three_panel_cm(horizon, mode)
        plt.show()
        plt.close(fig)